## Clustering

### Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

In [6]:
property_df = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')

print(f"Dataset shape: {property_df.shape}")
print(f"\nFirst few rows:")
property_df.head()

Dataset shape: (108372, 16)

First few rows:


,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Dist to CBD in Km,Sale Months Since Sep 2025,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded,District
0,-0.211237,-0.460443,0.774350,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,-1.619747,-1.228179,-0.334688,1.124971,1
1,-0.087630,-0.460443,1.240198,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.947931,-1.228179,-0.334688,1.124971,1
2,-0.303362,-0.331436,-0.058154,-0.618881,-1.117103,-0.723616,-0.407922,0.640686,0.276567,-0.286674,0.161507,-0.152502,0.814214,2.987854,-0.888912,1
3,-0.100129,-0.442004,1.103809,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.214309,-1.228179,-0.334688,1.124971,1
4,1.493910,2.267081,-0.950882,-0.363513,-1.117103,0.183604,-0.407922,0.156402,-0.670176,-0.286674,-0.030473,-1.619747,0.814214,-0.334688,-0.888912,1


### Helper functions

In [15]:
def discover_buyer_segments(property_df):
    cluster_features = [
        'Num MRT Within 1km',
        'Num Schools Within 2km',
        'Num Hospitals Within 5km',
        'Num Parks Within 1km',
        'Num Malls Within 1km',
        'Num Hawker Within 1km',
        'Dist to CBD in Km',
        'Floor_Level_Category',
        'Property_Type_Encoded',
        'Market_Segment_Encoded',
    ]
    
    X_cluster = property_df[cluster_features].copy()
    
    print(f"\nClustering based on {len(cluster_features)} features:")
    for i, feature in enumerate(cluster_features, 1):
        print(f"  {i:2d}. {feature}")
    
    optimal_k = 3
    
    print(f"\n{'='*80}")
    print(f"OPTIMAL K = {optimal_k}")
    print(f"{'='*80}")
    
    kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20, max_iter=300)
    property_df['Cluster'] = kmeans_final.fit_predict(X_cluster)
    
    print(f"\n✓ Properties assigned to {optimal_k} clusters")
    print(f"\nCluster distribution:")
    cluster_counts = property_df['Cluster'].value_counts().sort_index()
    for cluster_id, count in cluster_counts.items():
        pct = (count / len(property_df)) * 100
        print(f"  Cluster {cluster_id}: {count:5d} properties ({pct:5.1f}%)")
    
    return property_df, cluster_features, kmeans_final, X_cluster

In [16]:
def visualize_clusters_advanced(property_df, X_cluster, cluster_features):
    
    # 3D PCA Visualization
    
    print("\n3D PCA Visualization")
    
    pca_3d = PCA(n_components=3, random_state=42)
    X_pca_3d = pca_3d.fit_transform(X_cluster)
    
    # Explained variance
    explained_var = pca_3d.explained_variance_ratio_
    print(f"   PC1 explains {explained_var[0]*100:.1f}% of variance")
    print(f"   PC2 explains {explained_var[1]*100:.1f}% of variance")
    print(f"   PC3 explains {explained_var[2]*100:.1f}% of variance")
    print(f"   Total: {sum(explained_var)*100:.1f}%")
    
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']  # Red, Teal, Blue
    markers = ['o', 's', '^']  # Circle, Square, Triangle
    
    for cluster_id in sorted(property_df['Cluster'].unique()):
        cluster_mask = property_df['Cluster'] == cluster_id
        ax.scatter(
            X_pca_3d[cluster_mask, 0],
            X_pca_3d[cluster_mask, 1],
            X_pca_3d[cluster_mask, 2],
            c=colors[cluster_id],
            marker=markers[cluster_id],
            s=30,
            alpha=0.6,
            edgecolors='white',
            linewidth=0.5,
            label=f'Cluster {cluster_id} (n={cluster_mask.sum()})'
        )
    
    ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_zlabel(f'PC3 ({explained_var[2]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_title('3D Cluster Visualization (PCA Projection)\nProperty Segments in Feature Space', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    ax.view_init(elev=20, azim=45)
    
    plt.tight_layout()
    plt.savefig('clustering_outputs/cluster_3d_pca.png', dpi=200, bbox_inches='tight')
    print("Saved: cluster_3d_pca.png")
    plt.close()
    
    #2D PCA Pair Plots (all combinations)
    
    print("\n2D PCA Pair Plots (all combinations)")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    pairs = [(0, 1), (0, 2), (1, 2)]
    pair_labels = [
        (f'PC1 ({explained_var[0]*100:.1f}%)', f'PC2 ({explained_var[1]*100:.1f}%)'),
        (f'PC1 ({explained_var[0]*100:.1f}%)', f'PC3 ({explained_var[2]*100:.1f}%)'),
        (f'PC2 ({explained_var[1]*100:.1f}%)', f'PC3 ({explained_var[2]*100:.1f}%)')
    ]
    
    for idx, (pc1, pc2) in enumerate(pairs):
        ax = axes[idx]
        
        for cluster_id in sorted(property_df['Cluster'].unique()):
            cluster_mask = property_df['Cluster'] == cluster_id
            ax.scatter(
                X_pca_3d[cluster_mask, pc1],
                X_pca_3d[cluster_mask, pc2],
                c=colors[cluster_id],
                marker=markers[cluster_id],
                s=20,
                alpha=0.5,
                edgecolors='white',
                linewidth=0.3,
                label=f'Cluster {cluster_id}'
            )
        
        ax.set_xlabel(pair_labels[idx][0], fontsize=10)
        ax.set_ylabel(pair_labels[idx][1], fontsize=10)
        ax.set_title(f'{pair_labels[idx][0]} vs {pair_labels[idx][1]}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('clustering_outputs/cluster_2d_pca_pairs.png', dpi=150, bbox_inches='tight')
    print("Saved: cluster_2d_pca_pairs.png")
    plt.close()
    
    #Cluster Centroids in Original Feature Space
    
    print("\nCluster Centroids in Original Feature Space")
    
    # Calculate centroids
    centroids = []
    for cluster_id in sorted(property_df['Cluster'].unique()):
        cluster_mask = property_df['Cluster'] == cluster_id
        centroid = X_cluster[cluster_mask].mean(axis=0)
        centroids.append(centroid)
    
    centroids = np.array(centroids)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Heatmap of centroids
    feature_labels = [f.replace('Num ', '').replace(' Within', '\n') for f in cluster_features]
    
    sns.heatmap(
        centroids,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        xticklabels=feature_labels,
        yticklabels=[f'Cluster {i}' for i in range(len(centroids))],
        cbar_kws={'label': 'Normalized Value'},
        ax=ax
    )
    ax.set_title('Cluster Centroids in Original Feature Space\n(Higher = More of that amenity)', 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Features', fontsize=11)
    ax.set_ylabel('Clusters', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('clustering_outputs/cluster_centroids_heatmap.png', dpi=150, bbox_inches='tight')
    print("Saved: cluster_centroids_heatmap.png")
    plt.close()
    
    print("ALL CLUSTER VISUALIZATIONS COMPLETE")
    
    return X_pca_3d, pca_3d


def analyze_clusters(property_df, cluster_features):    
    n_clusters = property_df['Cluster'].nunique()
    overall_means = property_df[cluster_features].mean()
    cluster_profiles = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = property_df[property_df['Cluster'] == cluster_id]
        
        print(f"\n{'='*80}")
        print(f"CLUSTER {cluster_id} - {len(cluster_df)} properties ({len(cluster_df)/len(property_df)*100:.1f}%)")
        print(f"{'='*80}")
        
        cluster_means = cluster_df[cluster_features].mean()
        
        print(f"\n{'Feature':<35} {'Cluster Avg':>12} {'Overall Avg':>12} {'Difference':>12}")
        print("-" * 80)
        
        profile = {}
        
        for feature in cluster_features:
            cluster_val = cluster_means[feature]
            overall_val = overall_means[feature]
            diff = cluster_val - overall_val
            
            if abs(overall_val) > 0.01:
                pct_diff = ((cluster_val - overall_val) / abs(overall_val)) * 100
            else:
                pct_diff = 0
            
            profile[feature] = {
                'cluster_mean': cluster_val,
                'overall_mean': overall_val,
                'difference': diff,
                'pct_difference': pct_diff
            }
            
            if abs(diff) > 0.5:
                marker = "▲▲" if diff > 0 else "▼▼"
            elif abs(diff) > 0.25:
                marker = "▲ " if diff > 0 else "▼ "
            else:
                marker = "≈ "
            
            print(f"{feature:<35} {cluster_val:>12.3f} {overall_val:>12.3f} {marker} {diff:>+9.3f}")
        
        print(f"\n{'Price Statistics':}")
        print(f"  Median Transacted Price (normalized): {cluster_df['Transacted Price ($)'].median():>8.3f}")
        print(f"  Mean Transacted Price (normalized):   {cluster_df['Transacted Price ($)'].mean():>8.3f}")
        print(f"  Std Dev:                               {cluster_df['Transacted Price ($)'].std():>8.3f}")
        
        cluster_profiles[cluster_id] = profile
    
    return cluster_profiles


def calculate_pd_weights_per_cluster(property_df):

    amenity_features = [
        'Num MRT Within 1km',
        'Num Schools Within 2km',
        'Num Hospitals Within 5km',
        'Num Parks Within 1km',
        'Num Malls Within 1km',
        'Num Hawker Within 1km'
    ]
    
    n_clusters = property_df['Cluster'].nunique()
    cluster_weights = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = property_df[property_df['Cluster'] == cluster_id].copy()
        
        print(f"\n{'-'*80}")
        print(f"Cluster {cluster_id} - {len(cluster_df)} properties")
        print(f"{'-'*80}")
        
        X = cluster_df[amenity_features].values
        y = cluster_df['Transacted Price ($)'].values
        
        print(f"  Training Gradient Boosting model...")
        model = GradientBoostingRegressor(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            random_state=42,
            subsample=0.8
        )
        model.fit(X, y)
        
        train_score = model.score(X, y)
        print(f"  Model R² score: {train_score:.4f}")
        
        if train_score < 0.3:
            print(f"  ⚠ WARNING: Low R² score ({train_score:.4f}).")
        
        print(f"\n  Calculating Partial Dependence impacts...")
        impacts = {}
        
        for i, feature in enumerate(amenity_features):
            pd_result = partial_dependence(
                model, 
                X, 
                features=[i],
                grid_resolution=50
            )
            
            avg_predictions = pd_result['average'][0]
            impact = avg_predictions.max() - avg_predictions.min()
            impacts[feature] = impact
        
        total_impact = sum(impacts.values())
        
        if total_impact > 0:
            weights = {feature: impact / total_impact for feature, impact in impacts.items()}
        else:
            weights = {feature: 1.0 / len(amenity_features) for feature in amenity_features}
        
        cluster_weights[cluster_id] = {
            'impacts': impacts,
            'weights': weights,
            'model_r2': train_score
        }
        
        print(f"\n  Feature Importance (Partial Dependence Impact):")
        print(f"  {'Amenity':<35} {'Impact':>12} {'Weight':>10}")
        print(f"  {'-'*60}")
        
        sorted_features = sorted(weights.items(), key=lambda x: x[1], reverse=True)
        for feature, weight in sorted_features:
            impact = impacts[feature]
            print(f"  {feature:<35} {impact:>12.4f} {weight:>10.4f}")
    
    return cluster_weights, amenity_features


def link_clusters_to_age_groups(property_df, age_demographics_df):
    
    n_clusters = property_df['Cluster'].nunique()
    
    merged = property_df.merge(
        age_demographics_df,
        on='District',
        how='left'
    )
    
    cluster_age_profiles = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = merged[merged['Cluster'] == cluster_id]
        
        print(f"\n{'-'*80}")
        print(f"Cluster {cluster_id}")
        print(f"{'-'*80}")
        
        total_young = cluster_df['Young_Adults_20_35'].sum()
        total_families = cluster_df['Families_36_59'].sum()
        total_retirees = cluster_df['Retirees_60_Plus'].sum()
        
        total_pop = total_young + total_families + total_retirees
        
        if total_pop > 0:
            pct_young = (total_young / total_pop) * 100
            pct_families = (total_families / total_pop) * 100
            pct_retirees = (total_retirees / total_pop) * 100
        else:
            pct_young = pct_families = pct_retirees = 0
        
        print(f"\nAge Group Distribution in Cluster {cluster_id}'s Districts:")
        print(f"  Young Adults (20-35):  {pct_young:5.1f}%")
        print(f"  Families (36-59):      {pct_families:5.1f}%")
        print(f"  Retirees (60+):        {pct_retirees:5.1f}%")
        
        age_percentages = {
            'Young_Adults_20_35': pct_young,
            'Families_36_59': pct_families,
            'Retirees_60_Plus': pct_retirees
        }
        
        dominant_age_group = max(age_percentages, key=age_percentages.get)
        dominant_pct = age_percentages[dominant_age_group]
        
        print(f"\n  → Dominant Age Group: {dominant_age_group} ({dominant_pct:.1f}%)")
        
        cluster_age_profiles[cluster_id] = {
            'percentages': age_percentages,
            'dominant_group': dominant_age_group,
            'dominant_percentage': dominant_pct
        }
    
    return cluster_age_profiles


def assign_weights_to_age_groups(cluster_weights, cluster_age_profiles, amenity_features):
    
    age_group_preferences = {}
    
    for cluster_id, age_profile in cluster_age_profiles.items():
        age_group = age_profile['dominant_group']
        weights = cluster_weights[cluster_id]['weights']
        
        print(f"\nCluster {cluster_id} → {age_group}")
        
        if age_group not in age_group_preferences:
            age_group_preferences[age_group] = {'weights': weights, 'clusters': [cluster_id]}
        else:
            existing_weights = age_group_preferences[age_group]['weights']
            n_clusters = len(age_group_preferences[age_group]['clusters']) + 1
            
            averaged_weights = {}
            for feature in amenity_features:
                averaged_weights[feature] = (
                    existing_weights[feature] * (n_clusters - 1) + weights[feature]
                ) / n_clusters
            
            age_group_preferences[age_group]['weights'] = averaged_weights
            age_group_preferences[age_group]['clusters'].append(cluster_id)
    
    print(f"\n{'='*80}")
    print("FINAL AGE GROUP PREFERENCE WEIGHTS")
    print(f"{'='*80}")
    
    for age_group, data in age_group_preferences.items():
        print(f"\n{age_group}:")
        print(f"  (Derived from clusters: {data['clusters']})")
        print(f"\n  {'Amenity':<35} {'Weight':>10}")
        print(f"  {'-'*50}")
        
        sorted_weights = sorted(data['weights'].items(), key=lambda x: x[1], reverse=True)
        for feature, weight in sorted_weights:
            print(f"  {feature:<35} {weight:>10.4f}")
    
    return age_group_preferences



# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main(property_df, age_demographics_df):
    """
    Execute the full pipeline with enhanced visualizations.
    """
    
    print("\n" + "="*80)
    print("ENHANCED CLUSTER-BASED PREFERENCE DERIVATION PIPELINE")
    print("="*80)
    
    property_df, cluster_features, kmeans_model, X_cluster = discover_buyer_segments(property_df)
    cluster_weights, amenity_features = calculate_pd_weights_per_cluster(property_df)
    cluster_age_profiles = link_clusters_to_age_groups(property_df, age_demographics_df)
    age_group_preferences = assign_weights_to_age_groups(
        cluster_weights, 
        cluster_age_profiles, 
        amenity_features
    )
        
    print("PIPELINE COMPLETE")
    return age_group_preferences, cluster_weights, property_df



property_df = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')
age_demographics_df = pd.read_csv('../datasets/supplementary_datasets/demographics_by_district.csv')

age_group_preferences, cluster_weights, property_df_with_clusters = main(
    property_df, 
    age_demographics_df
)

print("FINAL OUTPUT: AGE GROUP PREFERENCE WEIGHTS")

for age_group, data in age_group_preferences.items():
    print(f"\n{age_group}:")
    for amenity, weight in data['weights'].items():
        print(f"  {amenity}: {weight:.4f}")


ENHANCED CLUSTER-BASED PREFERENCE DERIVATION PIPELINE

Clustering based on 10 features:
   1. Num MRT Within 1km
   2. Num Schools Within 2km
   3. Num Hospitals Within 5km
   4. Num Parks Within 1km
   5. Num Malls Within 1km
   6. Num Hawker Within 1km
   7. Dist to CBD in Km
   8. Floor_Level_Category
   9. Property_Type_Encoded
  10. Market_Segment_Encoded

OPTIMAL K = 3

✓ Properties assigned to 3 clusters

Cluster distribution:
  Cluster 0: 44177 properties ( 40.8%)
  Cluster 1: 40617 properties ( 37.5%)
  Cluster 2: 23578 properties ( 21.8%)

--------------------------------------------------------------------------------
Cluster 0 - 44177 properties
--------------------------------------------------------------------------------
  Training Gradient Boosting model...
  Model R² score: 0.2851
  ⚠ WARNING: Low R² score (0.2851).

  Calculating Partial Dependence impacts...

  Feature Importance (Partial Dependence Impact):
  Amenity                                   Impact     We